# 02_prototyping

In [22]:
from pathlib import Path
import pandas as pd
import shutil
import os
from sqlalchemy import create_engine, text
import psycopg2

Parameters

In [5]:
YEARS = [2011]

Read raw tables

In [17]:
data_dir = Path("../data/raw")
tables = {
    name.removesuffix(".csv"): pd.read_csv(data_dir / name)
    for name in [
        "calendar.csv",
        "sales_train_evaluation.csv",
        "sell_prices.csv",
    ]
}

Filter by year and save in interim

In [ ]:


sales_id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

for yi in YEARS:
    # 1. Define the path to data/interim/YEAR
    year_dir = Path(f"../data/interim/{yi}")
    
    # 2. Delete the directory if it already exists and recreate it clean
    if year_dir.exists():
        shutil.rmtree(year_dir)
    year_dir.mkdir(parents=True, exist_ok=True)
    
    # 3. Filter the tables
    df_cali = tables['calendar'][tables['calendar'].year == yi]
    day_id = df_cali.d
    wm_day_id = df_cali.wm_yr_wk

    df_sales = tables['sales_train_evaluation'][sales_id_cols + day_id.tolist()]
    df_prices = tables['sell_prices'][tables['sell_prices'].wm_yr_wk.isin(wm_day_id.tolist())]
    
    # 4. Save the tables in the corresponding year's directory
    # Note: We use index=False to avoid saving the Pandas index column
    df_cali.to_csv(year_dir / 'calendar.csv', index=False)
    df_sales.to_csv(year_dir / 'sales_train_evaluation.csv', index=False)
    df_prices.to_csv(year_dir / 'sell_prices.csv', index=False)
    
    print(f"✅ Tables for {yi} successfully saved in: {year_dir}")

✅ Tables for 2011 successfully saved in: ../data/interim/2011


Connect to database

In [2]:
db_user = os.environ.get("POSTGRES_USER", "prescriptai_user")
db_password = os.environ.get("POSTGRES_PASSWORD", "prescriptai_pass")
db_host = "localhost" 
db_port = os.environ.get("POSTGRES_PORT", "5432")
db_name = os.environ.get("POSTGRES_DB", "prescriptai_db")

In [3]:
# Construct the SQLAlchemy connection URI
database_uri = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

# 2. Create the database engine
engine = create_engine(database_uri)

print("✅ Database engine successfully created.")

✅ Database engine successfully created.


Upload tables to data base

In [ ]:
target_schema = "bronze"

for yi in YEARS:  # Use each year in the list
    year_dir = Path(f"../data/interim/{yi}")

    for table_name in [
        'calendar',
        'sales_train_evaluation',
        'sell_prices'
    ]:
        table_path = Path(year_dir / f"{table_name}.csv")
        df_i = pd.read_csv(table_path)

        print(f"Creating table '{table_name}' with {df_i.shape[1]} columns and {df_i.shape[0]} rows...")
            
        # .head(0) ensures only the schema (column names and data types) is sent to PostgreSQL
        target_table = f"{table_name}_{yi}"
        df_i.head(0).to_sql(
            name=target_table,
            con=engine,
            schema=target_schema,
            if_exists="replace", # Replaces the table if it already exists in the database
            index=False          # Prevents creating a separate column for the pandas index
        )
        
        print(f"✅ Table '{target_schema}.{target_table}' created successfully.")


        print(f"Uploading data to PostgreSQL ({target_schema}.{target_table})...")

        # Connect directly using psycopg2 to execute low-level streaming
        conn = psycopg2.connect(
            dbname=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port
        )

        with open(table_path, "r", encoding="utf-8") as f:
            # Read the header to extract column names
            header = f.readline().strip().split(",")
            columns = ", ".join([f'"{col}"' for col in header])
            f.seek(0)
            
            with conn.cursor() as cursor:
                copy_sql = f"COPY {target_schema}.{target_table} ({columns}) FROM STDIN WITH CSV HEADER"
                print(f"Streaming data directly into {target_table} via PostgreSQL COPY...")
                cursor.copy_expert(sql=copy_sql, file=f)
                conn.commit()

        conn.close()


        print(f"✅ {target_table} successfully uploaded to the database.")

Creating table 'calendar' with 14 columns and 337 rows...
✅ Table 'bronze.calendar_2011' created successfully.
Uploading data to PostgreSQL (bronze.calendar_2011)...
Streaming data directly into calendar_2011 via PostgreSQL COPY...
✅ calendar_2011 successfully uploaded to the database.
Creating table 'sales_train_evaluation' with 343 columns and 30490 rows...
✅ Table 'bronze.sales_train_evaluation_2011' created successfully.
Uploading data to PostgreSQL (bronze.sales_train_evaluation_2011)...
Streaming data directly into sales_train_evaluation_2011 via PostgreSQL COPY...
✅ sales_train_evaluation_2011 successfully uploaded to the database.
Creating table 'sell_prices' with 4 columns and 744086 rows...
✅ Table 'bronze.sell_prices_2011' created successfully.
Uploading data to PostgreSQL (bronze.sell_prices_2011)...
Streaming data directly into sell_prices_2011 via PostgreSQL COPY...
✅ sell_prices_2011 successfully uploaded to the database.


Existing tables

In [34]:
# list of existing tables in the 'bronze' schema
with engine.connect() as connection:
    query_tables = text("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'bronze'
        ORDER BY table_name;
    """)
    df_tables = pd.read_sql(query_tables, connection)
    
    print("\n=== EXISTING TABLES IN 'bronze' SCHEMA ===")
    display(df_tables)



=== EXISTING TABLES IN 'bronze' SCHEMA ===


,table_name
0,calendar_2011
1,sales_train_evaluation_2011
2,sell_prices_2011
